# YouTube Shorts Fully Automated Pipeline (Wav2Lip Avatar)

This notebook runs your entire YouTube automation pipeline in the cloud for free using a T4 GPU. Wav2Lip generates accurate lip-synced talking head videos with significantly better quality than SadTalker.

## Step 1: Mount Google Drive
This connects your Google Drive so the pipeline can save videos permanently.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Install Wav2Lip (GPU-Accelerated Lip Sync)
This clones Wav2Lip, downloads the model checkpoints (~160 MB), and patches the code for Python 3.12 and NumPy 2.0 compatibility.

**Why Wav2Lip over SadTalker?**
- Much more accurate lip sync, especially for Tamil audio
- Faster inference on Colab T4 GPU
- Smaller model footprint

In [ ]:
import os

WAV2LIP_DIR = '/content/Wav2Lip'

# Clone Wav2Lip
if not os.path.exists(WAV2LIP_DIR):
    !git clone https://github.com/Rudrabha/Wav2Lip.git {WAV2LIP_DIR}

%cd {WAV2LIP_DIR}

# Install modern-compatible dependencies (skip the pinned old versions)
!pip install -q librosa>=0.9.0 "opencv-contrib-python>=4.2.0,<4.10.0" "opencv-python>=4.2.0,<4.10.0" tqdm numba "numpy<2.0"

# Download model checkpoints
os.makedirs('checkpoints', exist_ok=True)

# wav2lip_gan.pth — best quality model
if not os.path.exists('checkpoints/wav2lip_gan.pth'):
    !wget -q -c "https://huggingface.co/camenduru/Wav2Lip/resolve/main/checkpoints/wav2lip_gan.pth" -O checkpoints/wav2lip_gan.pth
    print('✅ wav2lip_gan.pth downloaded')

# s3fd face detection model
os.makedirs('face_detection/detection/sfd', exist_ok=True)
if not os.path.exists('face_detection/detection/sfd/s3fd.pth'):
    !wget -q 'https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth' -O face_detection/detection/sfd/s3fd.pth
    print('✅ s3fd.pth downloaded')

# Patch for modern Python / NumPy compatibility
!sed -i 's/cv2.cv2.ROTATE_90_CLOCKWISE/cv2.ROTATE_90_CLOCKWISE/' inference.py

# Fix audio.py np.float deprecation if present
if os.path.exists('audio.py'):
    with open('audio.py', 'r') as f:
        content = f.read()
    if 'np.float' in content and 'np.float64' not in content:
        content = content.replace('np.float', 'float')
        content = content.replace('float32', 'np.float32')
        content = content.replace('float64', 'np.float64')
        with open('audio.py', 'w') as f:
            f.write(content)
        print('Patched audio.py for NumPy 2.0')

print('\n✅ Wav2Lip installation complete!')

## Step 3: Install Automation Dependencies
Make sure you have uploaded your `Automation` folder to the root of your Google Drive (`My Drive/Automation`).

In [ ]:
%cd /content/drive/MyDrive/Automation
# Remove specific version pins to avoid Colab conflicts, relying on Colab's pre-installed packages where possible
!sed -i 's/==.*//g' requirements.txt
!pip install -q -r requirements.txt

## Step 4: Run the Pipeline
You can run a specific day (`--generate-day 1`) or all days (`--generate-all`).

Use `--force` to regenerate cached content (e.g. to recreate avatar videos with Wav2Lip).

In [ ]:
# Generate a single day (use --force to regenerate the avatar)
!python main.py --generate-day 1 --force

## Step 5: Preview the Result
Play the generated video right here in the notebook.

In [ ]:
# Or generate all 30 days
# !python main.py --generate-all --force

In [ ]:
from IPython.display import HTML
from base64 import b64encode

video_path = '/content/drive/MyDrive/Automation/output/day_01/final_video.mp4'
mp4 = open(video_path, 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
HTML(f'''
<video width=360 controls>
  <source src="{data_url}" type="video/mp4">
</video>
''')